In [78]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

import numpy as np

In [79]:
num_epochs = 5
batch_size = 4
learning_rate = 0.001

In [80]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5,0.5, 0.5),(0.5,0.5,0.5))]
)

In [81]:
train_dataset= torchvision.datasets.CIFAR10(root='./data', train = True, download = True, transform= transform)

Files already downloaded and verified


In [82]:
test_dataset= torchvision.datasets.CIFAR10(root='./data', train = False, download = True, transform= transform)

Files already downloaded and verified


In [83]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, shuffle = True)

In [84]:
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [85]:
classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

In [86]:
class Convnet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,16,5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84,10)
        
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16*5*5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x


        

In [87]:
model = Convnet()

In [88]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr = learning_rate)

In [89]:
n_total_steps = len(train_loader)

for epoch in range(num_epochs):
    for i, (image,labels) in enumerate(train_loader):
        outputs = model(image)
        loss = criterion(outputs, labels)


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 2000 ==0:
            print(f'epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{n_total_steps}], Loss: {loss.item():.4f}')

    
print('finished training')

        
with torch.no_grad():
    n_correct = 0
    n_samples =0
    n_class_correct = [0 for i in range(10)]
    n_class_samples = [0 for i in range(10)]
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs,1)
        n_samples += labels.size(0)
        n_correct += (predicted == labels).sum().item()

        for i in range(batch_size):
            label = labels[i]
            pred = predicted[i]

            if (label == pred):
                n_class_correct[label] +=1

            n_class_samples[label] +=1

    acc = 100 * n_correct/ n_samples
    print(f'accuracy of the network: {acc}')

    for i in range(10):
        acc =100 * n_class_correct[i]/ n_class_samples[i]
        print(f'accuracy of {classes[i]}: {acc}' ) 



epoch [1/5], Step [2000/12500], Loss: 2.3116
epoch [1/5], Step [4000/12500], Loss: 2.2858
epoch [1/5], Step [6000/12500], Loss: 2.2869
epoch [1/5], Step [8000/12500], Loss: 2.1688
epoch [1/5], Step [10000/12500], Loss: 2.0308
epoch [1/5], Step [12000/12500], Loss: 1.9143
epoch [2/5], Step [2000/12500], Loss: 2.0011
epoch [2/5], Step [4000/12500], Loss: 1.5439
epoch [2/5], Step [6000/12500], Loss: 1.9293
epoch [2/5], Step [8000/12500], Loss: 1.8419
epoch [2/5], Step [10000/12500], Loss: 1.8344
epoch [2/5], Step [12000/12500], Loss: 1.5526
epoch [3/5], Step [2000/12500], Loss: 0.9789
epoch [3/5], Step [4000/12500], Loss: 1.5344
epoch [3/5], Step [6000/12500], Loss: 1.5825
epoch [3/5], Step [8000/12500], Loss: 1.1068
epoch [3/5], Step [10000/12500], Loss: 1.6507
epoch [3/5], Step [12000/12500], Loss: 1.4750
epoch [4/5], Step [2000/12500], Loss: 1.5404
epoch [4/5], Step [4000/12500], Loss: 2.0012
epoch [4/5], Step [6000/12500], Loss: 1.3050
epoch [4/5], Step [8000/12500], Loss: 1.5036
epoc